In [2]:
import os

PROJECT_DIR = os.path.abspath("BDNewsPaperScraper")
DB_PATH = os.path.join(PROJECT_DIR, "news_articles.db")

if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
    print("🗑️ Old DB deleted:", DB_PATH)
else:
    print("ℹ️ No existing DB found")

print("✅ Ready for fresh scrape")

ℹ️ No existing DB found
✅ Ready for fresh scrape


In [3]:
import os, sys, subprocess

REPO_URL = "https://github.com/EhsanulHaqueSiam/BDNewsPaperScraper.git"
PROJECT_DIR = os.path.abspath("BDNewsPaperScraper")

if not os.path.exists(PROJECT_DIR):
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
else:
    subprocess.run(["git", "-C", PROJECT_DIR, "pull"], check=True)

pkgs = [
    "scrapy>=2.12.0",
    "scrapy-playwright>=0.0.42",
    "playwright>=1.49.0",
    "pandas>=2.2.0",
    "openpyxl>=3.1.0",
    "pytz",
    "python-dateutil",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)
subprocess.run([sys.executable, "-m", "playwright", "install", "--with-deps", "chromium"], check=True)

print("✅ Setup done:", PROJECT_DIR)


✅ Setup done: /content/BDNewsPaperScraper


In [4]:
import subprocess, shlex

cmd = f"cd {shlex.quote(PROJECT_DIR)} && scrapy list"
print("Running:", cmd)
subprocess.run(["bash", "-lc", cmd], check=True)


Running: cd /content/BDNewsPaperScraper && scrapy list


CompletedProcess(args=['bash', '-lc', 'cd /content/BDNewsPaperScraper && scrapy list'], returncode=0)

In [5]:
from datetime import date

# 🔥 Modified for the FULL YEAR of 2024
start_date = date(2024, 1, 1)
end_date   = date(2024, 12, 31)

print("Scraping range set to:", start_date, "→", end_date)


Scraping range set to: 2024-01-01 → 2024-12-31


In [6]:
import time, subprocess, shlex

print("\n" + "="*80)
print(f"🚀 Running spider: thedailystar ({start_date} → {end_date})")

try:
    cmd = (
        f"cd {shlex.quote(PROJECT_DIR)} && "
        f"python -m scrapy crawl thedailystar -L INFO "
        f"-a start_date={start_date} -a end_date={end_date}"
    )

    code = subprocess.call(["bash", "-lc", cmd])

    if code != 0:
        print(f"❌ thedailystar FAILED (exit code {code})")
    else:
        print(f"✅ thedailystar DONE")

except Exception as e:
    print(f"❌ thedailystar CRASHED:", e)

time.sleep(3)
print("\n🎉 DONE — Daily Star scraping finished.")



🚀 Running spider: thedailystar (2024-01-01 → 2024-12-31)
✅ thedailystar DONE

🎉 DONE — Daily Star scraping finished.


In [7]:
import sqlite3, pandas as pd, os

DB_PATH = os.path.join(PROJECT_DIR, "news_articles.db")

with sqlite3.connect(DB_PATH) as conn:
    by_paper = pd.read_sql_query(
        """
        SELECT paper_name, COUNT(*) AS cnt
        FROM articles
        GROUP BY paper_name
        ORDER BY cnt DESC
        """,
        conn
    )

by_paper



,paper_name,cnt
0,thedailystar,383


In [8]:
import sqlite3, pandas as pd, os

DB_PATH = os.path.join(PROJECT_DIR, "news_articles.db")

with sqlite3.connect(DB_PATH) as conn:
    df = pd.read_sql_query(
        "SELECT * FROM articles WHERE paper_name = 'thedailystar'",
        conn
    )

print("✅ Total Daily Star articles:", len(df))
df.head(5)




✅ Total Daily Star articles: 383


,id,url,paper_name,headline,article,publication_date,scraped_at
0,1,https://www.thedailystar.net/star-multimedia/v...,thedailystar,"How could 30,000 foreigners stay here without ...","A report reveals at least 30,000 foreigners ar...","Tue Dec 24, 2024 01:27 AM",2025-12-20 13:50:37
1,2,https://www.thedailystar.net/star-multimedia/n...,thedailystar,Kings Party: A new frontier in Bangladesh's po...,"Recently, the term ""Kings Party"" has sparked d...","Mon Dec 30, 2024 09:29 PM",2025-12-20 13:50:41
2,3,https://www.thedailystar.net/star-multimedia/v...,thedailystar,‘King's Party’ tag still sticks on BNP: Samant...,"In this exclusive interview, Samantha Sharmin,...","Mon Dec 30, 2024 07:29 PM",2025-12-20 13:50:41
3,4,https://www.thedailystar.net/star-multimedia/b...,thedailystar,Businesses need a govt who can represent them:...,The interim government has passed 100 days sin...,"Sun Nov 24, 2024 12:00 AM",2025-12-20 13:50:44
4,5,https://www.thedailystar.net/star-multimedia/n...,thedailystar,Why is the govt repeatedly taking U-turns on d...,In the two months since the interim government...,"Wed Oct 9, 2024 10:35 PM",2025-12-20 13:50:44


In [9]:
# 🔎 URL column auto-detect
url_col = next(c for c in df.columns if "url" in c.lower() or "link" in c.lower())
print("Using URL column:", url_col)

# 🔥 Daily Star business URL keyword
BUSINESS_KEYWORDS = [
    "thedailystar.net/business"
]

urls = df[url_col].astype(str).str.lower()

finance_df = df[urls.str.contains("thedailystar.net/business", na=False)].copy()

print("✅ Daily Star Business articles:", len(finance_df))

finance_df[[url_col]].head(10)


Using URL column: url
✅ Daily Star Business articles: 2


,url
171,https://www.thedailystar.net/business/economy/...
195,https://www.thedailystar.net/business/economy/...


In [10]:
import sqlite3
import pandas as pd
import os
from google.colab import files

# 1. Connect and Load Data
DB_PATH = os.path.join(PROJECT_DIR, "news_articles.db")
with sqlite3.connect(DB_PATH) as conn:
    finance_df = pd.read_sql_query("SELECT * FROM articles", conn)

# 2. Convert date column to datetime objects
# Note: Your DB uses 'publication_date' based on your notebook
date_col = "publication_date"
finance_df[date_col] = pd.to_datetime(finance_df[date_col], errors="coerce")

# 3. Filter STRICTLY for year 2024
# This ensures even if the DB has other years, only 2024 is saved
finance_2024 = finance_df[finance_df[date_col].dt.year == 2024].copy()

# 4. Safety Check: Verify only 2024 exists in the final set
years_in_data = finance_2024[date_col].dt.year.unique()
print(f"📊 Years found in final file: {years_in_data}")

if finance_2024.empty:
    print("❌ No data found for 2024. Please check if the scraper finished successfully.")
else:
    # 5. Save and Download
    OUT = "dailystar_business_2024_ONLY.csv"
    finance_2024.to_csv(OUT, index=False, encoding="utf-8-sig")

    print("✅ Daily Star Business CSV (2024) saved at:", OUT)
    print("📄 Total rows:", len(finance_2024))
    files.download(OUT)


📊 Years found in final file: [2024]
✅ Daily Star Business CSV (2024) saved at: dailystar_business_2024_ONLY.csv
📄 Total rows: 383


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
from google.colab import files

OUT = "/content/dailystar_business_2024_January_Dec.csv"

if finance_df.empty:
    raise ValueError("❌ No Daily Star business data to export")

finance_df.to_csv(OUT, index=False, encoding="utf-8-sig")

print("✅ CSV saved:", OUT)
print("📄 Rows:", len(finance_df))

files.download(OUT)


✅ CSV saved: /content/dailystar_business_2024_January_Dec.csv
📄 Rows: 382


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
print("All columns:")
for c in finance_df.columns:
    print("-", c)

print("\nSample date values:")
for c in finance_df.columns:
    if "date" in c.lower() or "time" in c.lower():
        print(f"\nColumn: {c}")
        print(finance_df[c].dropna().head(5))


All columns:
- id
- url
- paper_name
- headline
- article
- publication_date
- scraped_at

Sample date values:

Column: publication_date
0   2024-12-24 01:27:00
1   2024-12-30 21:29:00
2   2024-12-30 19:29:00
3   2024-11-24 00:00:00
4   2024-10-09 22:35:00
Name: publication_date, dtype: datetime64[ns]


In [16]:
import re
import pandas as pd

def parse_date_safe(x):
    if pd.isna(x):
        return pd.NaT
    x = str(x)

    # Remove common prefixes
    x = re.sub(r"Published\s*:\s*", "", x, flags=re.I)
    x = re.sub(r"Updated\s*:\s*", "", x, flags=re.I)

    try:
        return pd.to_datetime(x)
    except:
        return pd.NaT

# 🔥 choose correct date column manually if needed
date_col = next(c for c in finance_df.columns if "date" in c.lower() or "time" in c.lower())
print("Using date column:", date_col)

finance_df[date_col] = finance_df[date_col].apply(parse_date_safe)

print("Parsed date sample:")
print(finance_df[date_col].dropna().head(5))


Using date column: publication_date
Parsed date sample:
0   2024-12-24 01:27:00
1   2024-12-30 21:29:00
2   2024-12-30 19:29:00
3   2024-11-24 00:00:00
4   2024-10-09 22:35:00
Name: publication_date, dtype: datetime64[ns]


In [15]:
finance_df = finance_df[
    (finance_df[date_col] >= pd.to_datetime(start_date)) &
    (finance_df[date_col] <= pd.to_datetime(end_date))
]

print("✅ Rows after date filter:", len(finance_df))

# sanity check
finance_df[[date_col]].describe()


✅ Rows after date filter: 382


,publication_date
count,382
mean,2024-06-11 07:31:16.649214720
min,2024-01-06 14:56:00
25%,2024-03-25 04:20:00
50%,2024-05-26 18:00:00
75%,2024-09-13 11:04:15
max,2024-12-30 21:29:00
